In [0]:
import pyspark.sql.functions as F
import pandas as pd
from datetime import date, timedelta

# Date boundaries
yesterday = (date.today() - timedelta(days=1)).strftime("%Y%m%d")

# Source tables
T_trans    = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_item     = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

In [0]:
games_categories = {
    "TRR - AMUSEMENT - G&E",
    "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES",
    "TRR - PROMO - AMUSEMENT",
}

food_categories = {
    "MISC",
    "PDM - MENU FOOD - CANDY",
    "PDM - MENU FOOD - G&E",
    "TRR - INV ALC BEV - BOTTLE",
    "TRR - INV ALC BEV - CAN",
    "TRR - INV ALC BEV - WINE",
    "TRR - MENU ALC BEV - COCKTAIL",
    "TRR - MENU ALC BEV - DRAUGHT BEER",
    "TRR - MENU BEV - NA COLD BEV",
    "TRR - MENU BEV - NA HOT BEV",
    "TRR - MENU FOOD - APP",
    "TRR - MENU FOOD - DESSERT",
    "TRR - MENU FOOD - G&E",
    "TRR - MENU FOOD - KIDS",
    "TRR - MENU FOOD - PIZZA",
    "TRR - MENU FOOD - POUTINE",
    "TRR - MENU FOOD - SANDWICHES",
    "TRR - MOD MENU - ADD-ON",
    "TRR - MOD MENU - NO VALUE",
    "CHARITY",
    "COUPONS",
    "TRR - INV BEV - NA BEV",
    "TRR - MENU ALC BEV - LIQUOR",
    "TRR - MENU ALC BEV - WINE",
    "TRR - MENU FOOD - ENTRÉE",
    "TRR - MENU FOOD - SALAD",
    "VHO-LICENSED FOOD",
    "Box Office Surcharges and Fees",
    "TRR - INV ALC BEV - LIQUOR",
    "VHO-CANDY",
    "VIP Food - Addons",
    "VHO-LICENSED DRINKS",
    "TRR - MENU FOOD - BRUNCH",
    "TRR - MENU FOOD - G&E OLD",
    "VHO-BIRTHDAY",
    "VHO-DRINK",
}

In [0]:
# Single source DataFrame for all downstream analyses (Jan 2025 → yesterday)
category_tx = (
    T_trans
    .join(T_location, T_trans.LocationId == T_location.LocationID, how="inner")
    .join(T_item, T_trans.ItemSkey == T_item.itemskey, how="inner")
    .filter(
        F.col("Location_Name").like("%Granvi%") |
        F.col("Location_Name").like("%Fairview%") |
        F.col("Location_Name").like("%Orion%") |
        F.col("Location_Name").like("%Whitby%")|
        F.col("Location_Name").like("%Royal%")
    )
    .filter(F.col("LC_Location_Type_Description").like("%Restaurant%"))
    .filter((F.col("VisitDateId") >= 20250101) & (F.col("VisitDateId") <= int(yesterday)))
    .filter(F.col("Item_Class_Description1").isin(list(games_categories) + list(food_categories)))
    .withColumn(
        "category_label",
        F.when(F.col("Item_Class_Description1").isin(list(games_categories)), F.lit("Games"))
        .when(F.col("Item_Class_Description1").isin(list(food_categories)), F.lit("Food"))
    )
    .filter(F.col("category_label").isNotNull())
)

In [0]:
# Monthly revenue aggregated at Spark level (single collect) — broken out by location
monthly_df = (
    category_tx
    .withColumn("year", F.year(F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd")))
    .withColumn("month", F.month(F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd")))
    .groupBy("Location_Name", "year", "month", "category_label")
    .agg(F.sum(F.col("Net_Revenue").cast("double")).alias("net_revenue"))
    .toPandas()
)

monthly_df["month_name"] = pd.to_datetime(monthly_df["month"], format="%m").dt.strftime("%B")
monthly_df = monthly_df.sort_values(["Location_Name", "category_label", "year", "month"]).reset_index(drop=True)

# --- YoY pivot per location ---
yoy_all = []
for loc in sorted(monthly_df["Location_Name"].unique()):
    loc_df = monthly_df[monthly_df["Location_Name"] == loc]
    yoy_loc = loc_df.pivot_table(
        index=["month", "month_name", "category_label"],
        columns="year",
        values="net_revenue"
    ).reset_index()
    year_cols = [c for c in yoy_loc.columns if isinstance(c, int)]
    if 2025 not in year_cols or 2026 not in year_cols:
        print(f"Skipping {loc} — data missing for one or both years")
        continue
    yoy_loc = yoy_loc.rename(columns={2025: "rev_2025", 2026: "rev_2026"})
    yoy_loc = yoy_loc.dropna(subset=["rev_2025", "rev_2026"])
    yoy_loc["yoy_change"] = yoy_loc["rev_2026"] - yoy_loc["rev_2025"]
    yoy_loc["yoy_pct"] = ((yoy_loc["rev_2026"] / yoy_loc["rev_2025"]) - 1) * 100
    yoy_loc["Location_Name"] = loc
    yoy_all.append(yoy_loc)
    print(f"=== YoY Performance — {loc} ===")
    display(yoy_loc.sort_values(["category_label", "month"]))

yoy_all = pd.concat(yoy_all, ignore_index=True) if yoy_all else pd.DataFrame()

In [0]:
# --- MoM: reuse monthly_df from Cell 4 ---
monthly_rev = monthly_df.copy()
monthly_rev["year_month"] = pd.to_datetime(
    monthly_rev["year"].astype(str) + "-" + monthly_rev["month"].astype(str).str.zfill(2)
)
monthly_rev = monthly_rev.sort_values(["Location_Name", "category_label", "year_month"]).reset_index(drop=True)

# MoM deltas — grouped by location + category
monthly_rev["prev_revenue"] = monthly_rev.groupby(["Location_Name", "category_label"])["net_revenue"].shift(1)
monthly_rev["mom_change"] = monthly_rev["net_revenue"] - monthly_rev["prev_revenue"]
monthly_rev["mom_pct"] = ((monthly_rev["net_revenue"] / monthly_rev["prev_revenue"]) - 1) * 100
monthly_rev["month_label"] = monthly_rev["year_month"].dt.strftime("%b %Y")

for loc in sorted(monthly_rev["Location_Name"].unique()):
    loc_mom = monthly_rev[monthly_rev["Location_Name"] == loc]
    print(f"=== MoM Revenue — {loc} ===")
    display(
        loc_mom.sort_values(["category_label", "year_month"])
        [["month_label", "category_label", "net_revenue", "mom_change", "mom_pct"]]
        .rename(columns={"month_label": "Month", "category_label": "Category",
                         "net_revenue": "Revenue", "mom_change": "MoM Δ ($)",
                         "mom_pct": "MoM Δ (%)"})
    )

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

COLORS    = {"Food": "#2196F3", "Games": "#FF9800"}
HIGHLIGHT = "#E91E63"
FIFA_COLOR = "#E91E63"

locations = sorted(monthly_rev["Location_Name"].unique())

# Global Y limits across all locations — keeps scales identical for direct comparison
yoy_max = max(yoy_all["rev_2025"].max(), yoy_all["rev_2026"].max()) * 1.15 if not yoy_all.empty else 1
mom_max = monthly_rev["net_revenue"].max() * 1.15

for loc in locations:
    yoy_loc = yoy_all[yoy_all["Location_Name"] == loc] if not yoy_all.empty else pd.DataFrame()
    mom_loc = monthly_rev[monthly_rev["Location_Name"] == loc]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"{loc} — Food & Games Revenue", fontsize=14, fontweight="bold", y=0.98)

    # ── TOP ROW: YoY Grouped Bars ──
    for idx, cat in enumerate(["Food", "Games"]):
        ax = axes[0, idx]
        sub = yoy_loc[yoy_loc["category_label"] == cat].sort_values("month")
        if sub.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{cat} — YoY Comparison", fontsize=11, fontweight="bold")
            continue
        x = np.arange(len(sub))
        w = 0.35
        bars_25 = ax.bar(x - w/2, sub["rev_2025"], w, label="2025", color="#B0BEC5", edgecolor="white")
        bars_26 = ax.bar(x + w/2, sub["rev_2026"], w, label="2026", color=COLORS[cat], edgecolor="white")

        june_mask = sub["month"].values == 6
        if june_mask.any():
            for j in x[june_mask]:
                bars_26[j].set_color(HIGHLIGHT)
                bars_26[j].set_edgecolor("black")
                bars_26[j].set_linewidth(1.5)
                pct = sub[sub["month"] == 6]["yoy_pct"].values[0]
                val = sub[sub["month"] == 6]["rev_2026"].values[0]
                ax.annotate(f"{pct:+.0f}% YoY", xy=(j + w/2, val),
                            xytext=(0, 8), textcoords="offset points",
                            ha="center", fontsize=9, fontweight="bold", color=HIGHLIGHT)

        ax.set_ylim(0, yoy_max)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["month_name"], rotation=45, ha="right", fontsize=8)
        ax.set_title(f"{cat} — YoY Comparison", fontsize=11, fontweight="bold")
        ax.set_ylabel("Revenue ($)")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v/1000:.0f}K"))
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)

    # ── BOTTOM ROW: MoM Trend Lines ──
    for idx, cat in enumerate(["Food", "Games"]):
        ax = axes[1, idx]
        sub = mom_loc[mom_loc["category_label"] == cat].sort_values("year_month")
        if sub.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{cat} — Monthly Revenue Trend (MoM)", fontsize=11, fontweight="bold")
            continue

        jan26 = pd.Timestamp("2026-01-01")
        ax.axvline(jan26, color="#455A64", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.text(jan26, mom_max * 0.97, " 2026 →", fontsize=7, color="#455A64",
                va="top", ha="left", fontstyle="italic")

        fifa_start = pd.Timestamp("2026-06-01")
        fifa_end   = pd.Timestamp("2026-06-30")
        ax.axvspan(fifa_start, fifa_end, alpha=0.10, color=FIFA_COLOR, zorder=0)
        ax.text(fifa_start + pd.Timedelta(days=15), mom_max * 0.03,
                "FIFA 2026", fontsize=7, ha="center", color=HIGHLIGHT,
                fontweight="bold", alpha=0.7)

        ax.plot(sub["year_month"], sub["net_revenue"], marker="o", markersize=5,
                color=COLORS[cat], linewidth=2, label=cat)
        ax.fill_between(sub["year_month"], sub["net_revenue"], alpha=0.06, color=COLORS[cat])

        june26 = sub[(sub["year"] == 2026) & (sub["month"] == 6)]
        if not june26.empty:
            jx  = june26["year_month"].values[0]
            jy  = june26["net_revenue"].values[0]
            mom = june26["mom_pct"].values[0]
            ax.scatter([jx], [jy], color=HIGHLIGHT, s=120, zorder=5,
                       edgecolors="black", linewidths=1.2)
            ax.annotate(f"Jun 2026\n${jy/1000:.0f}K ({mom:+.0f}% MoM)",
                        xy=(jx, jy), xytext=(20, 18), textcoords="offset points",
                        fontsize=8, fontweight="bold", color=HIGHLIGHT,
                        arrowprops=dict(arrowstyle="->", color=HIGHLIGHT, lw=1.2))

        ax.set_ylim(0, mom_max)
        ax.set_title(f"{cat} — Monthly Revenue Trend (MoM)", fontsize=11, fontweight="bold")
        ax.set_ylabel("Revenue ($)")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v/1000:.0f}K"))
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.tick_params(axis="x", rotation=45)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY: All-Locations Comparison on shared axes
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Distinct colour per location (consistent order)
_palette = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]  # red for the 5th location
LOC_COLORS = {loc: _palette[i] for i, loc in enumerate(sorted(monthly_rev["Location_Name"].unique()))}
HIGHLIGHT  = "#E91E63"
MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(2, 2, figsize=(18, 11), dpi=600)
fig.suptitle("All Locations — Food & Games Revenue Comparison",
             fontsize=14, fontweight="bold", y=0.99)

# ── TOP ROW: YoY % Change by Month (one line per location) ──
for idx, cat in enumerate(["Food", "Games"]):
    ax = axes[0, idx]
    sub_cat = yoy_all[yoy_all["category_label"] == cat]
    months_present = sorted(sub_cat["month"].unique())

    for loc in sorted(sub_cat["Location_Name"].unique()):
        loc_sub = sub_cat[sub_cat["Location_Name"] == loc].sort_values("month")
        ax.plot(loc_sub["month"], loc_sub["yoy_pct"], marker="o", markersize=6,
                color=LOC_COLORS[loc], linewidth=2, label=loc)
        june = loc_sub[loc_sub["month"] == 6]
        if not june.empty:
            ax.annotate(f"{june['yoy_pct'].values[0]:+.0f}%",
                        xy=(6, june["yoy_pct"].values[0]),
                        xytext=(5, 4), textcoords="offset points",
                        fontsize=7.5, fontweight="bold", color=LOC_COLORS[loc])

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
    if 6 in months_present:
        ax.axvline(6, color=HIGHLIGHT, linewidth=1.2, linestyle=":", alpha=0.6)
        ylims = ax.get_ylim()
        ax.text(6.08, ylims[1], "FIFA Jun", fontsize=7.5,
                color=HIGHLIGHT, fontstyle="italic", va="top")

    ax.set_xticks(months_present)
    ax.set_xticklabels([MONTH_NAMES[m - 1] for m in months_present],
                       rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{cat} — YoY % Change by Month", fontsize=11, fontweight="bold")
    ax.set_ylabel("YoY Change (%)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.0f}%"))
    ax.legend(fontsize=8, loc="best", framealpha=0.7)
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)

# ── BOTTOM ROW: MoM Revenue Trend (one line per location, shared Y = mom_max) ──
for idx, cat in enumerate(["Food", "Games"]):
    ax = axes[1, idx]

    for loc in sorted(monthly_rev["Location_Name"].unique()):
        sub = (monthly_rev[(monthly_rev["Location_Name"] == loc) &
                           (monthly_rev["category_label"] == cat)]
               .sort_values("year_month"))
        if sub.empty:
            continue
        ax.plot(sub["year_month"], sub["net_revenue"], marker="o", markersize=4,
                color=LOC_COLORS[loc], linewidth=2, label=loc, alpha=0.9)

    jan26 = pd.Timestamp("2026-01-01")
    ax.axvline(jan26, color="#455A64", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.text(jan26, mom_max * 0.97, " 2026 →", fontsize=7, color="#455A64",
            va="top", ha="left", fontstyle="italic")

    ax.axvspan(pd.Timestamp("2026-06-01"), pd.Timestamp("2026-06-30"),
               alpha=0.10, color=HIGHLIGHT, zorder=0)
    ax.text(pd.Timestamp("2026-06-15"), mom_max * 0.03, "FIFA 2026",
            fontsize=7, ha="center", color=HIGHLIGHT, fontweight="bold", alpha=0.7)

    ax.set_ylim(0, mom_max)   # same global max as per-location charts
    ax.set_title(f"{cat} — Monthly Revenue by Location", fontsize=11, fontweight="bold")
    ax.set_ylabel("Revenue ($)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v/1000:.0f}K"))
    ax.legend(fontsize=8, loc="upper left", framealpha=0.7)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    ax.tick_params(axis="x", rotation=45)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL A — FIFA INCREMENTALITY: Pre-trend-adjusted DiD (exact tournament window)
# Window: Jun 11 – Jul 19 (both years) | Pre-period: Jan–May (both years)
# Lift = (Food−Games YoY in window) − (Food−Games YoY in pre-period)
# ─────────────────────────────────────────────────────────────────────────────

# Tournament window + pre-period, as YYYYMMDD ints (matches VisitDateId)
WIN_25 = (20250611, 20250719)
WIN_26 = (20260611, 20260719)
PRE_25 = (20250101, 20250531)
PRE_26 = (20260101, 20260531)

def _period_rev(df_spark, lo, hi):
    """Sum Net_Revenue by location + category over an inclusive VisitDateId range."""
    return (
        df_spark
        .filter((F.col("VisitDateId") >= lo) & (F.col("VisitDateId") <= hi))
        .groupBy("Location_Name", "category_label")
        .agg(F.sum(F.col("Net_Revenue").cast("double")).alias("rev"))
        .toPandas()
    )

# Pull the four period totals (reuses category_tx from your earlier cell)
win25 = _period_rev(category_tx, *WIN_25)
win26 = _period_rev(category_tx, *WIN_26)
pre25 = _period_rev(category_tx, *PRE_25)
pre26 = _period_rev(category_tx, *PRE_26)

def _pivot(df, tag):
    p = df.pivot_table(index="Location_Name", columns="category_label",
                       values="rev", aggfunc="sum")
    for c in ["Food", "Games"]:
        if c not in p.columns:
            p[c] = float("nan")
    return p[["Food", "Games"]].rename(columns={"Food": f"Food_{tag}", "Games": f"Games_{tag}"})

did = (_pivot(win25, "w25").join(_pivot(win26, "w26"), how="outer")
                           .join(_pivot(pre25, "p25"), how="outer")
                           .join(_pivot(pre26, "p26"), how="outer"))

# YoY deltas
did["Food_win_yoy"]  = did["Food_w26"]  - did["Food_w25"]
did["Games_win_yoy"] = did["Games_w26"] - did["Games_w25"]
did["Food_pre_yoy"]  = did["Food_p26"]  - did["Food_p25"]
did["Games_pre_yoy"] = did["Games_p26"] - did["Games_p25"]

# DiD gaps
did["raw_did"]      = did["Food_win_yoy"] - did["Games_win_yoy"]   # 2-period DiD
did["pretrend_gap"] = did["Food_pre_yoy"] - did["Games_pre_yoy"]   # standing Food-vs-Games drift
did["adj_lift"]     = did["raw_did"] - did["pretrend_gap"]         # FIFA-attributable, trend-adjusted

# Control-contamination check: Games YoY % in the window
did["Games_win_yoy_pct"] = (did["Games_w26"] / did["Games_w25"] - 1) * 100

out = did[["raw_did", "pretrend_gap", "adj_lift", "Games_win_yoy", "Games_win_yoy_pct"]].round(0)
out = out.rename(columns={
    "raw_did":           "Raw DiD ($)",
    "pretrend_gap":      "Pre-trend gap ($)",
    "adj_lift":          "Adjusted lift ($)",
    "Games_win_yoy":     "Games YoY in window ($)",
    "Games_win_yoy_pct": "Games YoY in window (%)",
})
print("=== CELL A — Pre-trend-adjusted DiD (Jun 11 – Jul 19) ===")
print("Adjusted lift = FIFA-attributable incremental F&B, per location.")
print("Watch 'Games YoY in window': large negative = control may be contaminated (viewers not playing).\n")
display(out.sort_values("Adjusted lift ($)", ascending=False))
print(f"\nTotal adjusted incremental F&B across locations: ${out['Adjusted lift ($)'].sum():,.0f}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL B — FIFA INCREMENTALITY: Monthly regression DiD (per location)
# Model per location:  net_revenue ~ is_food + is_june2026 + is_food×is_june2026
#                                     + month fixed effects + year term
# The interaction coefficient = incremental Food lift in June 2026, net of trend.
# NOTE: monthly granularity → window approximates to June, not exact Jun 11–Jul 19.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd

reg = monthly_df.copy()   # from your earlier cell: Location_Name, year, month, category_label, net_revenue
reg["is_food"]       = (reg["category_label"] == "Food").astype(int)
reg["is_june2026"]   = ((reg["year"] == 2026) & (reg["month"] == 6)).astype(int)
reg["food_x_fifa"]   = reg["is_food"] * reg["is_june2026"]
reg["year_c"]        = reg["year"] - reg["year"].min()   # linear year trend

# Month fixed effects (drop month 1 as reference)
month_dummies = pd.get_dummies(reg["month"], prefix="m", drop_first=True).astype(float)

def _ols(X, y):
    """Plain OLS with SEs. Uses statsmodels if present, else numpy fallback."""
    try:
        import statsmodels.api as sm
        m = sm.OLS(y, X).fit()
        return m.params, m.bse, m.pvalues
    except ModuleNotFoundError:
        Xv = X.values.astype(float); yv = y.values.astype(float)
        XtX_inv = np.linalg.pinv(Xv.T @ Xv)
        beta = XtX_inv @ Xv.T @ yv
        resid = yv - Xv @ beta
        dof = max(len(yv) - Xv.shape[1], 1)
        sigma2 = (resid @ resid) / dof
        se = np.sqrt(np.diag(sigma2 * XtX_inv))
        from scipy import stats
        pvals = 2 * (1 - stats.t.cdf(np.abs(beta / se), dof))
        idx = X.columns
        return pd.Series(beta, idx), pd.Series(se, idx), pd.Series(pvals, idx)

rows = []
for loc in sorted(reg["Location_Name"].unique()):
    d = reg[reg["Location_Name"] == loc].copy()
    md = month_dummies.loc[d.index]
    X = pd.concat([
        pd.Series(1.0, index=d.index, name="const"),
        d[["is_food", "is_june2026", "food_x_fifa", "year_c"]].astype(float),
        md,
    ], axis=1)
    y = d["net_revenue"].astype(float)
    if len(d) < X.shape[1] + 2:   # not enough rows to identify the model
        rows.append({"Location_Name": loc, "Incremental lift ($)": np.nan,
                     "SE ($)": np.nan, "p-value": np.nan, "n_obs": len(d), "note": "insufficient data"})
        continue
    params, bse, pvals = _ols(X, y)
    rows.append({
        "Location_Name":       loc,
        "Incremental lift ($)": round(float(params["food_x_fifa"]), 0),
        "SE ($)":               round(float(bse["food_x_fifa"]), 0),
        "p-value":              round(float(pvals["food_x_fifa"]), 3),
        "n_obs":                len(d),
        "note":                 "",
    })

reg_out = pd.DataFrame(rows).set_index("Location_Name")
print("=== CELL B — Monthly regression DiD (June 2026 interaction) ===")
print("'Incremental lift' = Food revenue in June 2026 above what month/year/category trend predicts.")
print("Compare against Cell A's 'Adjusted lift'. Same ballpark = robust. Divergence = investigate.\n")
display(reg_out)
print(f"\nTotal incremental F&B (regression): "
      f"${reg_out['Incremental lift ($)'].sum(skipna=True):,.0f}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL C3 — Counterfactual-gap incrementality, formatted for reading
# Same computation as C2; cleaner presentation.
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

m = monthly_df.copy()

def _rev(df, loc, cat, yr, mo):
    v = df[(df.Location_Name==loc)&(df.category_label==cat)&(df.year==yr)&(df.month==mo)]["net_revenue"]
    return float(v.iloc[0]) if len(v) else np.nan

rows = []
for loc in sorted(m["Location_Name"].unique()):
    yoy_pre = [(_rev(m,loc,"Food",2026,mo)/_rev(m,loc,"Food",2025,mo) - 1)
               for mo in range(1,6)
               if _rev(m,loc,"Food",2025,mo) and not np.isnan(_rev(m,loc,"Food",2026,mo))]
    trend = np.mean(yoy_pre) if yoy_pre else np.nan
    g_pre = [(_rev(m,loc,"Games",2026,mo)/_rev(m,loc,"Games",2025,mo) - 1)
             for mo in range(1,6)
             if _rev(m,loc,"Games",2025,mo) and not np.isnan(_rev(m,loc,"Games",2026,mo))]
    games_trend = np.mean(g_pre) if g_pre else np.nan

    for mo, label in [(6, "June (full)"), (7, "July (partial)")]:
        f25, f26 = _rev(m, loc, "Food", 2025, mo), _rev(m, loc, "Food", 2026, mo)
        cf   = f25 * (1 + trend) if not np.isnan(trend) and not np.isnan(f25) else np.nan
        incr = f26 - cf if not np.isnan(cf) else np.nan
        g25, g26 = _rev(m, loc, "Games", 2025, mo), _rev(m, loc, "Games", 2026, mo)
        g_yoy = (g26/g25 - 1) if g25 and not np.isnan(g26) else np.nan
        clean = (abs(g_yoy - games_trend) <= 0.10) if not np.isnan(g_yoy) and not np.isnan(games_trend) else None

        rows.append({
            "Location": loc.replace("The Rec Room ", "").replace("Rec Room ", ""),
            "Period": label,
            "Counterfactual": cf,
            "Actual": f26,
            "YoY": (f26/f25-1) if f25 and not np.isnan(f26) else np.nan,
            "Incremental F&B": incr,
            "Games YoY": g_yoy,
            "Control": {True:"✓ clean", False:"✗ deviates", None:"—"}[clean],
        })

res = pd.DataFrame(rows)

# --- formatting ---
def _money(v):  return "—" if pd.isna(v) else f"${v:,.0f}"
def _pct(v):    return "—" if pd.isna(v) else f"{v*100:+.0f}%"

fmt = res.copy()
for c in ["Counterfactual", "Actual", "Incremental F&B"]:
    fmt[c] = res[c].map(_money)
for c in ["YoY", "Games YoY"]:
    fmt[c] = res[c].map(_pct)

styled = (
    fmt.style
       .hide(axis="index")
       .set_caption("FIFA Incrementality — Counterfactual-Gap Method (per location)")
       .set_properties(**{"text-align": "right", "font-size": "12px"})
       .set_properties(subset=["Location", "Period", "Control"], **{"text-align": "left"})
       .set_table_styles([
           {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold"),("padding","6px")]},
           {"selector": "th", "props": [("text-align","left"),("background-color","#F5F5F5"),("font-size","12px")]},
       ])
       # shade June rows (the clean, headline period) lightly
       .apply(lambda r: ["background-color:#FCE4EC" if "June" in r["Period"] else "" for _ in r], axis=1)
)
display(styled)

print("June rows shaded = headline period (full month). July = partial, understated.")
print("'Control ✓ clean' = June Games tracked its own Jan–May trend → lift is F&B-specific.")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL D2 — Detrending visual, June + July counterfactual gaps (5 locations)
# Two counterfactual points per panel (Jun, Jul). July gap dashed lighter = partial.
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

m = monthly_df.copy()
FOOD_C, REF_C, HL = "#2196F3", "#B0BEC5", "#E91E63"
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

def _rev(df, loc, cat, yr, mo):
    v = df[(df.Location_Name==loc)&(df.category_label==cat)&(df.year==yr)&(df.month==mo)]["net_revenue"]
    return float(v.iloc[0]) if len(v) else np.nan

locs = sorted(m["Location_Name"].unique())
fig, axes = plt.subplots(2, 3, figsize=(22, 10))
fig.suptitle("FIFA Incrementality — Actual vs Counterfactual Food (June + July gaps)",
             fontsize=14, fontweight="bold", y=0.98)

for ax, loc in zip(axes.flat, locs):
    f25 = [_rev(m, loc, "Food", 2025, mo) for mo in range(1, 13)]
    f26 = [_rev(m, loc, "Food", 2026, mo) for mo in range(1, 13)]
    yoy_pre = [(_rev(m,loc,"Food",2026,mo)/_rev(m,loc,"Food",2025,mo) - 1)
               for mo in range(1,6)
               if _rev(m,loc,"Food",2025,mo) and not np.isnan(_rev(m,loc,"Food",2026,mo))]
    trend = np.mean(yoy_pre) if yoy_pre else np.nan

    x = np.arange(1, 13)
    ax.plot(x, f25, marker="o", ms=4, color=REF_C, lw=1.8, label="2025 Food", zorder=2)
    ax.plot(x, f26, marker="o", ms=4, color=FOOD_C, lw=2, label="2026 Food (actual)", zorder=3)

    for mo, partial in [(6, False), (7, True)]:
        cf = f25[mo-1] * (1 + trend) if not np.isnan(trend) and not np.isnan(f25[mo-1]) else np.nan
        act = f26[mo-1]
        if np.isnan(cf) or np.isnan(act):
            continue
        ax.scatter([mo], [cf], color="white", edgecolor=HL, s=85, lw=2, zorder=5)
        ax.scatter([mo], [act], color=HL, s=85, zorder=5, edgecolor="black", lw=1)
        ax.plot([mo, mo], [cf, act], color=HL, lw=2.2,
                ls=":" if partial else "--", alpha=0.6 if partial else 1.0, zorder=4)
        incr = act - cf
        ax.annotate(f"${incr/1000:+.0f}K" + ("*" if partial else ""),
                    xy=(mo, (cf+act)/2), xytext=(6, 0), textcoords="offset points",
                    fontsize=8.5, fontweight="bold", color=HL, va="center",
                    alpha=0.7 if partial else 1.0)

    ttl = loc + (f"   (trend {trend*100:+.0f}%)" if not np.isnan(trend) else "")
    ax.set_title(ttl, fontsize=11, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(MONTHS, fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"${v/1000:.0f}K"))
    ax.set_ylabel("Food Revenue ($)")
    ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
    ax.axvspan(5.5, 7.5, alpha=0.05, color=HL, zorder=0)
    ax.legend(fontsize=7.5, loc="upper left", framealpha=0.8)

# hide any unused panels (6 slots, 5 locations)
for ax in axes.flat[len(locs):]:
    ax.set_visible(False)

fig.text(0.5, 0.005, "* July gap is partial-month (data through yesterday) — understated vs full-month 2025 counterfactual.",
         ha="center", fontsize=8, fontstyle="italic", color=HL)
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL C4 — Blended incremental F&B by location (June + July combined)
# Same counterfactual-gap computation as C2/C3. Totals only; caveat July yourself.
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

m = monthly_df.copy()

def _rev(df, loc, cat, yr, mo):
    v = df[(df.Location_Name==loc)&(df.category_label==cat)&(df.year==yr)&(df.month==mo)]["net_revenue"]
    return float(v.iloc[0]) if len(v) else np.nan

rows = []
for loc in sorted(m["Location_Name"].unique()):
    yoy_pre = [(_rev(m,loc,"Food",2026,mo)/_rev(m,loc,"Food",2025,mo) - 1)
               for mo in range(1,6)
               if _rev(m,loc,"Food",2025,mo) and not np.isnan(_rev(m,loc,"Food",2026,mo))]
    trend = np.mean(yoy_pre) if yoy_pre else np.nan

    def _incr(mo):
        f25, f26 = _rev(m, loc, "Food", 2025, mo), _rev(m, loc, "Food", 2026, mo)
        cf = f25 * (1 + trend) if not np.isnan(trend) and not np.isnan(f25) else np.nan
        return (f26 - cf) if not np.isnan(cf) and not np.isnan(f26) else np.nan

    jun, jul = _incr(6), _incr(7)
    rows.append({
        "Location":            loc.replace("The Rec Room ", "").replace("Rec Room ", ""),
        "June incremental":    jun,
        "July incremental":    jul,
        "Blended (Jun+Jul)":   np.nansum([jun, jul]) if not (np.isnan(jun) and np.isnan(jul)) else np.nan,
    })

res = pd.DataFrame(rows)

# Grand-total row across locations
total = pd.DataFrame([{
    "Location": "ALL LOCATIONS",
    "June incremental":  res["June incremental"].sum(skipna=True),
    "July incremental":  res["July incremental"].sum(skipna=True),
    "Blended (Jun+Jul)": res["Blended (Jun+Jul)"].sum(skipna=True),
}])
res = pd.concat([res, total], ignore_index=True)

# formatting
def _money(v): return "—" if pd.isna(v) else f"${v:,.0f}"
fmt = res.copy()
for c in ["June incremental", "July incremental", "Blended (Jun+Jul)"]:
    fmt[c] = res[c].map(_money)

styled = (
    fmt.style
       .hide(axis="index")
       .set_caption("FIFA Incremental F&B — Blended by Location (June + July)")
       .set_properties(**{"text-align": "right", "font-size": "12px"})
       .set_properties(subset=["Location"], **{"text-align": "left"})
       .set_table_styles([
           {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold"),("padding","6px")]},
           {"selector": "th", "props": [("text-align","left"),("background-color","#F5F5F5")]},
       ])
       .apply(lambda r: ["font-weight:bold;background-color:#ECEFF1" if r["Location"]=="ALL LOCATIONS" else "" for _ in r], axis=1)
)
display(styled)

## FIFA F&B Hypothesis Assessment — Detrended Counterfactual Method

**Method:** Counterfactual = Food_June_2025 × (1 + Jan–May Food YoY trend). Incremental = Actual − Counterfactual.  
**Control:** Games YoY in June vs its own Jan–May trend (within ±10% = "clean").

| Location | Finding | Assessment |
| --- | --- | --- |
| Fairview | Food trend Jan–May: +21.8%. Counterfactual: $49,299 → Actual: $36,870. **Incremental: −$12,429.** Food underperformed its own growth trend in June. Games YoY −35.6% vs trend −31.2% (control ✓ clean). | **DOES NOT SUPPORT** |
| Orion Gate | Food trend Jan–May: +16.3%. Counterfactual: $48,797 → Actual: $51,999. **Incremental: +$3,202** (negligible). Games YoY −10.7% vs trend +0.5% (control ✗ deviates — FIFA likely cannibalized Games, contaminating baseline). | **DOES NOT SUPPORT** |
| Whitby | Food trend Jan–May: +5.3%. Counterfactual: $58,444 → Actual: $75,642. **Incremental: +$17,198.** Food beat detrended expectation. Games YoY −16.2% vs trend −16.9% (control ✓ clean — lift is F&B-specific). | **SUPPORTS** |
| Granville | Food trend Jan–May: −27.6% (declining). Counterfactual: $240,809 → Actual: $584,439. **Incremental: +$343,629.** Massive reversal of declining trend. Games YoY −28.0% vs trend −34.2% (control ✓ clean). Regression p=0.013. | **STRONGLY SUPPORTS** |
| Royal Mount | Food trend Jan–May: +25.5%. Counterfactual: $172,306 → Actual: $194,322. **Incremental: +$22,015.** Modest beat over trend. Games YoY +4.8% vs trend −0.4% (control ✓ clean). | **WEAKLY SUPPORTS** |

**Summary:** 3 of 5 locations show positive FIFA-attributable F&B lift after detrending. Granville dominates ($344K, statistically significant). Fairview shows no effect; Orion Gate is inconclusive with a contaminated control.

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# EXPANDED: All Restaurant-type locations, segmented by LC_Trade_Brand_Description
# ─────────────────────────────────────────────────────────────────────────────
import pyspark.sql.functions as F
import pandas as pd
from datetime import date, timedelta

yesterday = (date.today() - timedelta(days=1)).strftime("%Y%m%d")

T_trans    = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_item     = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

# Same category sets as the original analysis
games_categories = {
    "TRR - AMUSEMENT - G&E", "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES", "TRR - PROMO - AMUSEMENT",
}
food_categories = {
    "MISC", "PDM - MENU FOOD - CANDY", "PDM - MENU FOOD - G&E",
    "TRR - INV ALC BEV - BOTTLE", "TRR - INV ALC BEV - CAN", "TRR - INV ALC BEV - WINE",
    "TRR - MENU ALC BEV - COCKTAIL", "TRR - MENU ALC BEV - DRAUGHT BEER",
    "TRR - MENU BEV - NA COLD BEV", "TRR - MENU BEV - NA HOT BEV",
    "TRR - MENU FOOD - APP", "TRR - MENU FOOD - DESSERT", "TRR - MENU FOOD - G&E",
    "TRR - MENU FOOD - KIDS", "TRR - MENU FOOD - PIZZA", "TRR - MENU FOOD - POUTINE",
    "TRR - MENU FOOD - SANDWICHES", "TRR - MOD MENU - ADD-ON", "TRR - MOD MENU - NO VALUE",
    "CHARITY", "COUPONS", "TRR - INV BEV - NA BEV", "TRR - MENU ALC BEV - LIQUOR",
    "TRR - MENU ALC BEV - WINE", "TRR - MENU FOOD - ENTRÉE", "TRR - MENU FOOD - SALAD",
    "VHO-LICENSED FOOD", "Box Office Surcharges and Fees", "TRR - INV ALC BEV - LIQUOR",
    "VHO-CANDY", "VIP Food - Addons", "VHO-LICENSED DRINKS", "TRR - MENU FOOD - BRUNCH",
    "TRR - MENU FOOD - G&E OLD", "VHO-BIRTHDAY", "VHO-DRINK",
}

# Build expanded DataFrame — ALL restaurant locations, with Trade Brand
category_tx_all = (
    T_trans
    .join(T_location, T_trans.LocationId == T_location.LocationID, how="inner")
    .join(T_item, T_trans.ItemSkey == T_item.itemskey, how="inner")
    .filter(F.col("LC_Location_Type_Description").like("Restaurant%"))
    .filter((F.col("VisitDateId") >= 20250101) & (F.col("VisitDateId") <= int(yesterday)))
    .filter(F.col("Item_Class_Description1").isin(list(games_categories) + list(food_categories)))
    .withColumn(
        "category_label",
        F.when(F.col("Item_Class_Description1").isin(list(games_categories)), F.lit("Games"))
        .when(F.col("Item_Class_Description1").isin(list(food_categories)), F.lit("Food"))
    )
    .filter(F.col("category_label").isNotNull())
)

# Monthly aggregation by Trade Brand + Location
monthly_brand = (
    category_tx_all
    .withColumn("year", F.year(F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd")))
    .withColumn("month", F.month(F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd")))
    .groupBy("LC_Trade_Brand_Description", "Location_Name", "year", "month", "category_label")
    .agg(F.sum(F.col("Net_Revenue").cast("double")).alias("net_revenue"))
    .toPandas()
)

monthly_brand = monthly_brand.sort_values(
    ["LC_Trade_Brand_Description", "Location_Name", "category_label", "year", "month"]
).reset_index(drop=True)

print(f"Locations: {monthly_brand['Location_Name'].nunique()}")
print(f"Trade Brands: {monthly_brand['LC_Trade_Brand_Description'].nunique()}")
print("\nLocations per brand:")
display(monthly_brand.groupby("LC_Trade_Brand_Description")["Location_Name"].nunique().reset_index(name="num_locations"))

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# FIFA Incrementality — Counterfactual-Gap Method, segmented by Trade Brand
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

m = monthly_brand.copy()

def _rev(df, loc, cat, yr, mo):
    v = df[(df.Location_Name==loc)&(df.category_label==cat)&(df.year==yr)&(df.month==mo)]["net_revenue"]
    return float(v.iloc[0]) if len(v) else np.nan

rows = []
for loc in sorted(m["Location_Name"].unique()):
    brand = m[m.Location_Name == loc]["LC_Trade_Brand_Description"].iloc[0]
    # Jan-May Food YoY trend
    yoy_pre = [(_rev(m,loc,"Food",2026,mo)/_rev(m,loc,"Food",2025,mo) - 1)
               for mo in range(1,6)
               if _rev(m,loc,"Food",2025,mo) and not np.isnan(_rev(m,loc,"Food",2026,mo))]
    trend = np.mean(yoy_pre) if yoy_pre else np.nan
    # Games trend for control check
    g_pre = [(_rev(m,loc,"Games",2026,mo)/_rev(m,loc,"Games",2025,mo) - 1)
             for mo in range(1,6)
             if _rev(m,loc,"Games",2025,mo) and not np.isnan(_rev(m,loc,"Games",2026,mo))]
    games_trend = np.mean(g_pre) if g_pre else np.nan

    for mo, label in [(6, "June"), (7, "July")]:
        f25, f26 = _rev(m, loc, "Food", 2025, mo), _rev(m, loc, "Food", 2026, mo)
        cf   = f25 * (1 + trend) if not np.isnan(trend) and not np.isnan(f25) else np.nan
        incr = f26 - cf if not np.isnan(cf) and not np.isnan(f26) else np.nan
        g25, g26 = _rev(m, loc, "Games", 2025, mo), _rev(m, loc, "Games", 2026, mo)
        g_yoy = (g26/g25 - 1) if g25 and not np.isnan(g26) else np.nan
        clean = (abs(g_yoy - games_trend) <= 0.10) if not np.isnan(g_yoy) and not np.isnan(games_trend) else None

        rows.append({
            "Trade Brand": brand,
            "Location": loc,
            "Period": label,
            "Food Trend (Jan-May)": trend,
            "Counterfactual": cf,
            "Actual": f26,
            "Incremental F&B": incr,
            "Games YoY": g_yoy,
            "Control": {True: "✓ clean", False: "✗ deviates", None: "—"}[clean],
        })

res_brand = pd.DataFrame(rows)

# ── Brand-level summary (June + July blended) ──
brand_summary = (
    res_brand.groupby(["Trade Brand", "Location", "Period"])["Incremental F&B"]
    .sum().reset_index()
    .pivot_table(index=["Trade Brand", "Location"], columns="Period",
                 values="Incremental F&B", aggfunc="sum")
    .reset_index()
)
brand_summary["Blended (Jun+Jul)"] = brand_summary[["June", "July"]].sum(axis=1, skipna=True)

# Grand totals per brand
brand_totals = brand_summary.groupby("Trade Brand")[["June", "July", "Blended (Jun+Jul)"]].sum().reset_index()
brand_totals["Location"] = "— BRAND TOTAL —"

# Grand total all
grand_total = pd.DataFrame([{
    "Trade Brand": "ALL BRANDS",
    "Location": "— GRAND TOTAL —",
    "June": brand_summary["June"].sum(skipna=True),
    "July": brand_summary["July"].sum(skipna=True),
    "Blended (Jun+Jul)": brand_summary["Blended (Jun+Jul)"].sum(skipna=True),
}])

full_table = pd.concat([brand_summary, brand_totals, grand_total], ignore_index=True)
full_table = full_table.sort_values(["Trade Brand", "Location"]).reset_index(drop=True)

# Format
def _money(v): return "—" if pd.isna(v) else f"${v:,.0f}"
fmt = full_table.copy()
for c in ["June", "July", "Blended (Jun+Jul)"]:
    fmt[c] = full_table[c].map(_money)

styled = (
    fmt.style
       .hide(axis="index")
       .set_caption("FIFA Incremental F&B — All Restaurant Locations by Trade Brand")
       .set_properties(**{"text-align": "right", "font-size": "11px"})
       .set_properties(subset=["Trade Brand", "Location"], **{"text-align": "left"})
       .set_table_styles([
           {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold"),("padding","6px")]},
           {"selector": "th", "props": [("text-align","left"),("background-color","#F5F5F5"),("font-size","11px")]},
       ])
       .apply(lambda r: ["font-weight:bold;background-color:#ECEFF1" if "TOTAL" in str(r["Location"]) else "" for _ in r], axis=1)
)
display(styled)

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Visual: Incremental F&B by Trade Brand (bar chart)
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# Use brand_totals from previous cell
bt = brand_totals.copy()
bt = bt[bt["Trade Brand"] != "ALL BRANDS"].sort_values("Blended (Jun+Jul)", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
colors = {"Rec Room": "#2196F3", "Playdium": "#4CAF50", "Outtakes": "#FF9800"}
bar_colors = [colors.get(b, "#9E9E9E") for b in bt["Trade Brand"]]

y_pos = np.arange(len(bt))
bars = ax.barh(y_pos, bt["Blended (Jun+Jul)"], color=bar_colors, edgecolor="white", height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(bt["Trade Brand"], fontsize=11)
ax.set_xlabel("Incremental F&B ($)", fontsize=10)
ax.set_title("FIFA Incremental F&B by Trade Brand (Jun + Jul Blended)", fontsize=12, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v/1000:,.0f}K"))

# Value labels
for bar, val in zip(bars, bt["Blended (Jun+Jul)"]):
    offset = 5 if val >= 0 else -5
    ha = "left" if val >= 0 else "right"
    ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", ha=ha, fontsize=9, fontweight="bold")

ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Detailed: June-only results per location, grouped by Trade Brand (with control)
# ─────────────────────────────────────────────────────────────────────────────
june_detail = res_brand[res_brand["Period"] == "June"].copy()
june_detail = june_detail.sort_values(["Trade Brand", "Incremental F&B"], ascending=[True, False])

def _money(v): return "—" if pd.isna(v) else f"${v:,.0f}"
def _pct(v):   return "—" if pd.isna(v) else f"{v*100:+.0f}%"

fmt_detail = june_detail.copy()
fmt_detail["Food Trend (Jan-May)"] = june_detail["Food Trend (Jan-May)"].map(_pct)
fmt_detail["Counterfactual"] = june_detail["Counterfactual"].map(_money)
fmt_detail["Actual"] = june_detail["Actual"].map(_money)
fmt_detail["Incremental F&B"] = june_detail["Incremental F&B"].map(_money)
fmt_detail["Games YoY"] = june_detail["Games YoY"].map(_pct)
fmt_detail = fmt_detail.drop(columns=["Period"])

styled_detail = (
    fmt_detail.style
       .hide(axis="index")
       .set_caption("FIFA Incrementality — June 2026 Detail by Trade Brand & Location")
       .set_properties(**{"text-align": "right", "font-size": "11px"})
       .set_properties(subset=["Trade Brand", "Location", "Control"], **{"text-align": "left"})
       .set_table_styles([
           {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold"),("padding","6px")]},
           {"selector": "th", "props": [("text-align","left"),("background-color","#F5F5F5"),("font-size","11px")]},
       ])
)
display(styled_detail)

print(f"\nLocations with positive June lift: {(june_detail['Incremental F&B'] > 0).sum()} / {len(june_detail)}")
print(f"Locations with clean control: {(june_detail['Control'] == '✓ clean').sum()} / {len(june_detail)}")